# Limpeza de `tickets_suporte.csv`

**Objetivo:** gerar os chamados ao suporte prontos para análise em `dados/processados/tickets_suporte_limpa.parquet`.

**Regras aplicadas** (decisões de 07/11/2026; o diagnóstico está no notebook 20):

| Passo | Regra |
|---|---|
| 1 | `aberto_em` como data e hora em Brasília |
| 2 | Status no corte: `status_no_corte` e `resolvido_no_corte`; os da exportação ficam como `status_exportacao` e `resolvido_em_exportacao` |
| 3 | `categoria_do_texto` (a mais frequente entre os chamados com o mesmo texto) e a marca `categoria_diverge` |
| 4 | `sobre_falha_gps`: chamados sobre GPS, percurso ou treino perdido (o salto de maio e junho de 2026) |

**Atenção ao usar:** use `status_no_corte`, não o da exportação. A `categoria_atendente` tem ruído (32% divergem do texto):
prefira a `categoria_do_texto` ou a extração com IA generativa. Nas análises de chamados, separe o período da falha de GPS.

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.

In [1]:
ARQUIVO = "tickets_suporte.csv"
NOME_SAIDA = "tickets_suporte_limpa.parquet"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df, alteradas=0):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df),
                     "alteradas": int(alteradas)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas, {int(alteradas):,} alteradas)")
    return df

SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)
    return t.isin(SEM_CONTEUDO) | so_simbolo

usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

## 0. Leitura

In [2]:
tk = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(tk) == 4_422 and tk["ticket_id"].is_unique
base = registrar("0. Arquivo bruto", "tickets_suporte.csv sem filtro", tk.copy())

0. Arquivo bruto: 4,422 linhas (0 removidas, 0 alteradas)


## 1. Data e hora de abertura

In [3]:
base["aberto_em"] = pd.to_datetime(base["aberto_em"], format="%Y-%m-%d %H:%M:%S")
assert base["aberto_em"].notna().all() and (base["aberto_em"] < FIM_CORTE).all()
base = registrar("1. Abertura em Brasília", "aberto_em como horário de Brasília", base, len(base))

1. Abertura em Brasília: 4,422 linhas (0 removidas, 4,422 alteradas)


## 2. Status no corte

O status e a data de resolução são os da exportação. No corte, um chamado só estava resolvido se a resolução veio até
30/09/2026; os resolvidos depois voltam a `aberto`.

In [4]:
base["status_exportacao"] = base["status"]
base["resolvido_em_exportacao"] = pd.to_datetime(base["resolvido_em"].replace("", None), format="%Y-%m-%d")
depois = base["resolvido_em_exportacao"] > CORTE
base["status_no_corte"] = base["status_exportacao"].where(~depois, "aberto")
base["resolvido_no_corte"] = base["resolvido_em_exportacao"].where(~depois)
display(pd.crosstab(base["status_exportacao"], base["status_no_corte"]))
assert depois.sum() == 41
assert ((base["status_no_corte"] == "resolvido") == base["resolvido_no_corte"].notna()).all()
base = registrar("2. Status no corte", "resolvido só se resolvido_em <= 30/09/2026; senão aberto", base, depois.sum())

status_no_corte,aberto,resolvido,sem_resposta
status_exportacao,,,
aberto,981,0,0
resolvido,41,2876,0
sem_resposta,0,0,524


2. Status no corte: 4,422 linhas (0 removidas, 41 alteradas)


## 3. Categoria do texto

In [5]:
moda = base.groupby("texto_cliente")["categoria_atendente"].agg(lambda s: s.value_counts().index[0])
base["categoria_do_texto"] = base["texto_cliente"].map(moda)
base["categoria_diverge"] = base["categoria_atendente"] != base["categoria_do_texto"]
print(f"divergem: {base['categoria_diverge'].sum():,} ({100 * base['categoria_diverge'].mean():.0f}%)")
assert base["categoria_diverge"].sum() == 1_419
base = registrar("3. Categoria do texto", "categoria mais frequente do mesmo texto; marca categoria_diverge", base,
                 base["categoria_diverge"].sum())

divergem: 1,419 (32%)
3. Categoria do texto: 4,422 linhas (0 removidas, 1,419 alteradas)


## 4. Chamados da falha de GPS

In [6]:
GPS = r"gps|percurso|rota|distância|distancia|travou|perdi meus treinos"
base["sobre_falha_gps"] = base["texto_cliente"].str.lower().str.contains(GPS, regex=True)
por_mes = base.groupby(base["aberto_em"].dt.to_period("M"))["sobre_falha_gps"].sum()
print("chamados sobre GPS, percurso ou treino perdido por mês:", {str(k): int(v) for k, v in por_mes.items() if v})
assert set(str(k) for k, v in por_mes.items() if v) == {"2026-05", "2026-06"}
base = registrar("4. Falha de GPS", "marca sobre_falha_gps", base)

chamados sobre GPS, percurso ou treino perdido por mês: {'2026-05': 675, '2026-06': 516}
4. Falha de GPS: 4,422 linhas (0 removidas, 0 alteradas)


## 5. Conferências

In [7]:
conferencias = {
    "linhas": (len(base), 4_422),
    "ticket_id repetido": (int(base["ticket_id"].duplicated().sum()), 0),
    "contas fora de usuarios_limpa": (int((~base["user_id"].isin(usuarios.index)).sum()), 0),
    "chamado antes do dia do cadastro": (int((base["aberto_em"].dt.normalize() < base["user_id"].map(usuarios["data_cadastro"])).sum()), 0),
    "resolvido no corte depois do corte": (int((base["resolvido_no_corte"] > CORTE).sum()), 0),
    "resolvido antes de aberto": (int((base["resolvido_no_corte"] < base["aberto_em"].dt.normalize()).sum()), 0),
}

In [8]:
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()

,obtido,esperado,ok
linhas,4422,4422,True
ticket_id repetido,0,0,True
contas fora de usuarios_limpa,0,0,True
chamado antes do dia do cadastro,0,0,True
resolvido no corte depois do corte,0,0,True
resolvido antes de aberto,0,0,True


## 6. Saídas e integridade dos brutos

In [9]:
COLUNAS = ["ticket_id", "user_id", "aberto_em", "canal", "texto_cliente", "categoria_atendente", "categoria_do_texto",
           "categoria_diverge", "status_no_corte", "resolvido_no_corte", "status_exportacao", "resolvido_em_exportacao",
           "sobre_falha_gps"]
limpa = base.sort_values(["user_id", "aberto_em", "ticket_id"])[COLUNAS].reset_index(drop=True)

for col in limpa.columns[limpa.dtypes == object]:
    limpa[col] = limpa[col].astype(object).where(limpa[col].notna(), None)   # ausente sempre como None

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / NOME_SAIDA
limpa.to_parquet(destino, index=False)
relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)
pd.DataFrame(registro).to_csv(SAIDAS / f"limpeza_{Path(ARQUIVO).stem}_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\tickets_suporte_limpa.parquet: 4,422 linhas x 13 colunas


,tipo,ausentes
ticket_id,object,0
user_id,object,0
aberto_em,datetime64[ns],0
canal,object,0
texto_cliente,object,0
categoria_atendente,object,0
categoria_do_texto,object,0
categoria_diverge,bool,0
status_no_corte,object,0
resolvido_no_corte,datetime64[ns],1546


,passo,regra,linhas,removidas,alteradas
0,0. Arquivo bruto,tickets_suporte.csv sem filtro,4422,0,0
1,1. Abertura em Brasília,aberto_em como horário de Brasília,4422,0,4422
2,2. Status no corte,resolvido só se resolvido_em <= 30/09/2026; senão aberto,4422,0,41
3,3. Categoria do texto,categoria mais frequente do mesmo texto; marca categoria_diverge,4422,0,1419
4,4. Falha de GPS,marca sobre_falha_gps,4422,0,0


dados/brutos/ inalterado: True
